# Gemma 3 轻量大模型推理优化与质量评测

运行前请在 Colab 中选择 GPU，并在 Hugging Face 接受 `google/gemma-3-1b-it` 的许可。随后在 Colab Secrets 中创建名为 `HF_TOKEN` 的密钥。不要把 Token 写进代码或提交到 GitHub。

In [ ]:
REPO_URL = 'https://github.com/zmh2245749337/gemma-inference-eval.git'
!git clone {REPO_URL}
%cd gemma-inference-eval
!pip -q install -r requirements-colab.txt

In [ ]:
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if not token:
    raise ValueError('没有读取到 HF_TOKEN，请在 Colab Secrets 中添加。')
os.environ['HF_TOKEN'] = token
!nvidia-smi

## 1. 手写解码与 Hugging Face 对齐

先使用贪心解码，检查两种实现是否逐 token 一致。

In [ ]:
!python scripts/run_manual_decode.py --prompt '请用三句话解释什么是KV Cache。' --max-new-tokens 48 --greedy --check-hf-parity --output reports/manual_decode.json

## 2. FP16 KV Cache 长度基准

固定提示词，分别生成 32、64 和 128 tokens；每组自动比较开启和关闭 Cache。

In [ ]:
from pathlib import Path
Path('reports/benchmark.csv').unlink(missing_ok=True)
for length in (32, 64, 128):
    !python scripts/run_benchmark.py --precision fp16 --max-new-tokens {length} --warmups 1 --repeats 3 --output reports/benchmark.csv

## 3. 四位量化 KV Cache 长度基准

使用同一提示词、长度、预热次数和重复次数，保证与 FP16 可比。

In [ ]:
for length in (32, 64, 128):
    !python scripts/run_benchmark.py --precision 4bit --max-new-tokens {length} --warmups 1 --repeats 3 --output reports/benchmark.csv

## 4. 固定题集质量检查

不同精度应使用不同输出文件，避免覆盖证据。

In [ ]:
for output in ('reports/quality_fp16.jsonl', 'reports/quality_4bit.jsonl'):
    Path(output).unlink(missing_ok=True)
!python scripts/run_quality_eval.py --precision fp16 --dataset configs/prompts_zh.jsonl --output reports/quality_fp16.jsonl
!python scripts/run_quality_eval.py --precision 4bit --dataset configs/prompts_zh.jsonl --output reports/quality_4bit.jsonl

In [ ]:
import pandas as pd
benchmark = pd.read_csv('reports/benchmark.csv')
display(benchmark[['precision', 'use_kv_cache', 'generated_tokens', 'ttft_ms_median', 'decode_tps_median', 'peak_memory_mb']])
fp16 = pd.read_json('reports/quality_fp16.jsonl', lines=True).set_index('case_id')
bit4 = pd.read_json('reports/quality_4bit.jsonl', lines=True).set_index('case_id')
comparison = pd.DataFrame({
    'expected': fp16['must_contain'],
    'fp16_answer': fp16['answer'],
    'fp16_pass': fp16['pass'],
    '4bit_answer': bit4['answer'],
    '4bit_pass': bit4['pass'],
})
display(comparison[comparison['fp16_pass'] != comparison['4bit_pass']])
!python scripts/plot_results.py --results-dir reports

## 5. 下载原始证据

将 CSV、JSONL、环境记录和图表一起下载，供报告复核。

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/gemma_t4_results', 'zip', root_dir='reports')
files.download(archive)